In [2]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

TRANS_PATH = r"C:\Users\muaad\Downloads\ieee-fraud-detection\train_transaction.csv"
ID_PATH    = r"C:\Users\muaad\Downloads\ieee-fraud-detection\train_identity.csv"

CHUNK_SIZE = 50000
SEQ_LEN = 5


In [4]:
identity_df = pd.read_csv(ID_PATH)


In [5]:
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])
    return np.array(X_seq), np.array(y_seq)


In [6]:
client1_X, client1_y = [], []
client2_X, client2_y = [], []
scaler = StandardScaler()

for chunk in pd.read_csv(TRANS_PATH, chunksize=CHUNK_SIZE):

    merged = chunk.merge(identity_df, on="TransactionID", how="left")
    merged = merged.fillna(0)

    y = merged["isFraud"].values.astype("int8")
    X = merged.drop(["isFraud", "TransactionID"], axis=1)
    X = X.select_dtypes(include=[np.number]).astype("float32").values
    X = scaler.fit_transform(X)


    if len(X) <= SEQ_LEN:
        continue

    X_seq, y_seq = create_sequences(X, y, SEQ_LEN)

    mid = len(X_seq) // 2
    client1_X.append(X_seq[:mid])
    client1_y.append(y_seq[:mid])
    client2_X.append(X_seq[mid:])
    client2_y.append(y_seq[mid:])



In [7]:
np.savez(
    "../client_sequences/client_1_seq.npz",
    X=np.vstack(client1_X),
    y=np.hstack(client1_y)
)

np.savez(
    "../client_sequences/client_2_seq.npz",
    X=np.vstack(client2_X),
    y=np.hstack(client2_y)
)

print("✅ Transaction + Identity merged and saved")


✅ Transaction + Identity merged and saved


In [8]:
data = np.load("../client_sequences/client_1_seq.npz")
print(data["X"].shape, data["y"].shape)


(295234, 5, 401) (295234,)


In [9]:
print(
    np.load("../client_sequences/client_1_seq.npz")["X"].shape,
    np.load("../client_sequences/client_2_seq.npz")["X"].shape
)


(295234, 5, 401) (295246, 5, 401)
